# Final Submission - Option 1: Quantitative Evaluation

For the final submission, the project uses **Option 1: Quantitative Evaluation**.  
This option is appropriate because the final submission brief explicitly allows custom metrics such as **precision@k** and **recall@k**, and the project already satisfies the dataset scaling requirement with **701,092 records** in the cleaned All Beauty dataset. Therefore, the final addition focuses on evaluating retrieval quality quantitatively rather than rescaling the corpus again.

## 1. Imports

In [ ]:
from pathlib import Path
import pandas as pd
from typing import Any

from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever

from src.preprocessing import find_repo_root
from src.utils.io import load_reviews
from src.utils.retriever_loading import load_saved_retrievers

from src.utils.evaluation import (
    collect_results,
    evaluate_results,
    summarize_evaluation,
)

## 2. Evaluation goal

The goal of this section is to evaluate retrieval quality quantitatively using **precision@k** and **recall@k**.  
These are appropriate metrics for this project because they allow us to compare how well BM25, semantic retrieval, and hybrid retrieval return relevant documents for the same query set.

- **precision@k** measures how many of the top-k retrieved documents are relevant
- **recall@k** measures how many of the relevant documents are successfully retrieved in the top-k

In [2]:
# Define project directories

PROJECT_ROOT = find_repo_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / Path("notebooks/outputs")
RESULTS_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR =", DATA_DIR)
print("RESULTS_DIR =", RESULTS_DIR)

PROJECT_ROOT = c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray
DATA_DIR = c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data
RESULTS_DIR = c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\notebooks\outputs


In [3]:
# Define path to the processed data
DATA_PATH = DATA_DIR / "processed" / "All_Beauty_clean.parquet"
DATA_PATH

WindowsPath('c:/Users/ruthy/assignments/block6/group_575/DSCI_575_project_omo001_deepray/data/processed/All_Beauty_clean.parquet')

We then load the cleaned retrieval dataset that was prepared earlier in the project workflow. Using the saved cleaned file keeps the notebook reproducible, avoids repeating preprocessing inside the analysis, and ensures that all later retrieval and evaluation steps use the same standardized All Beauty dataset.

In [4]:
df = load_reviews(DATA_PATH)
df.head()

,doc_id,parent_asin,asin,title,rating,text
0,B00YQ6X8EO_0,B00YQ6X8EO,B00YQ6X8EO,Such a lovely scent but not overpowering.,5,such a lovely scent but not overpowering. this...
1,B081TJ8YS3_1,B081TJ8YS3,B081TJ8YS3,Works great but smells a little weird.,4,works great but smells a little weird. this pr...
2,B097R46CSY_2,B097R46CSY,B07PNNCSP9,Yes!,5,"yes! smells good, feels great!"
3,B09JS339BZ_3,B09JS339BZ,B09JS339BZ,Synthetic feeling,1,synthetic feeling felt synthetic
4,B08BZ63GMJ_4,B08BZ63GMJ,B08BZ63GMJ,A+,5,a+ love it


In [5]:
df.shape

(701092, 6)

## 3. Query set

The evaluation reuses five representative queries from the existing project workflow. This keeps the quantitative evaluation aligned with the earlier retrieval and RAG experiments and makes comparisons easier to interpret.

In [6]:
evaluation_queries_df = pd.DataFrame([
    {"query_id": 1, "query": "face moisturizer", "difficulty": "easy"},
    {"query_id": 2, "query": "something for dry skin", "difficulty": "medium"},
    {"query_id": 3, "query": "product to reduce frizzy hair", "difficulty": "medium"},
    {"query_id": 4, "query": "beauty product that is easy to carry while traveling", "difficulty": "complex"},
    {"query_id": 5, "query": "makeup remover that does not irritate sensitive skin", "difficulty": "complex"},
])

evaluation_queries_df

,query_id,query,difficulty
0,1,face moisturizer,easy
1,2,something for dry skin,medium
2,3,product to reduce frizzy hair,medium
3,4,beauty product that is easy to carry while tra...,complex
4,5,makeup remover that does not irritate sensitiv...,complex


## 4. Load the existing retrievers

The retrieval artifacts were already built and saved earlier in the project workflow, so the notebook loads those persisted retrievers directly rather than rebuilding them. This keeps the evaluation workflow efficient, avoids unnecessary recomputation, and ensures that the notebook uses the same retrieval setup as the rest of the project.

In [7]:

bm25_retriever, semantic_retriever, hybrid_retriever = load_saved_retrievers(PROJECT_ROOT)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2748.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 5. Collect retrieval results for evaluation

To support quantitative evaluation, retrieval outputs are first collected in a long-format table. This avoids repeating retrieval calls across later steps and makes it easier to inspect results, assign relevance labels, and compare BM25, semantic, and hybrid retrieval consistently.

In [8]:
bm25_results_df = collect_results(
    retriever=bm25_retriever,
    queries_df=evaluation_queries_df,
    method_name="BM25",
    top_k=5,
)

semantic_results_df = collect_results(
    retriever=semantic_retriever,
    queries_df=evaluation_queries_df,
    method_name="Semantic",
    top_k=5,
)

hybrid_results_df = collect_results(
    retriever=hybrid_retriever,
    queries_df=evaluation_queries_df,
    method_name="Hybrid",
    top_k=5,
)

all_results_df = pd.concat(
    [bm25_results_df, semantic_results_df, hybrid_results_df],
    ignore_index=True,
)

In [9]:

all_results_df.head()

,method,query_id,query,difficulty,rank,doc_id,title,rating,score,text
0,BM25,1,face moisturizer,easy,1,B01MRR1SAS_591326,Nice lightweight face moisturizer,5,14.030204,nice lightweight face moisturizer nice lightwe...
1,BM25,1,face moisturizer,easy,2,B007HLWRTM_154324,Great moisturizer!,5,13.844107,great moisturizer! great moisturizer that does...
2,BM25,1,face moisturizer,easy,3,B07X9VJCN9_622049,Great Face Moisturizer!,4,13.731127,great face moisturizer! good! this face moistu...
3,BM25,1,face moisturizer,easy,4,B07MX673RB_241638,Good product,5,13.658535,good product face moisturizer
4,BM25,1,face moisturizer,easy,5,B08XMBHL8C_92790,Moisturizer,3,13.508304,moisturizer love the face moisturizer but this...


## 6. Collect retrieval outputs for all methods

To avoid repeating nearly identical code for BM25, semantic retrieval, and hybrid retrieval, this section uses a single loop over the three retrievers. This keeps the notebook cleaner, reduces duplication, and makes the evaluation workflow easier to maintain and reproduce.

In [10]:
def collect_labeling_view(
    results_df: pd.DataFrame,
    queries_df: pd.DataFrame,
    methods: list[str] | None = None,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    """Collect retrieval results for manual relevance labeling.

    Parameters
    ----------
    results_df : pd.DataFrame
        Long-format retrieval results dataframe.
    queries_df : pd.DataFrame
        Query dataframe containing at least a `query` column.
    methods : list of str or None, default=None
        Retrieval methods to include. If None, all methods are used.
    columns : list of str or None, default=None
        Columns to keep in the returned dataframe.

    Returns
    -------
    pd.DataFrame
        Filtered dataframe for manual relevance labeling.
    """
    if methods is None:
        methods = sorted(results_df["method"].dropna().unique().tolist())

    if columns is None:
        columns = ["method", "query", "rank", "doc_id", "title", "rating", "score", "text"]

    query_values = queries_df["query"].tolist()

    filtered = results_df.loc[
        results_df["method"].isin(methods) & results_df["query"].isin(query_values),
        columns,
    ].copy()

    return filtered.sort_values(["query", "method", "rank"]).reset_index(drop=True)

In [11]:
# Collect labeling view dataframe for manual relevance labeling
labeling_view_df = collect_labeling_view(
    results_df=all_results_df,
    queries_df=evaluation_queries_df,
    methods=["BM25", "Semantic", "Hybrid"],
)

labeling_view_df.head(5)

,method,query,rank,doc_id,title,rating,score,text
0,BM25,beauty product that is easy to carry while tra...,1,B07BR19ML2_51821,Love the variety,5,23.086363,love the variety this manicure set is very goo...
1,BM25,beauty product that is easy to carry while tra...,2,B085XQTBXP_330271,Handy sanitizing wipe,4,21.883126,handy sanitizing wipe 75% alcohol wipes that c...
2,BM25,beauty product that is easy to carry while tra...,3,B005FMUBLG_205504,Five Stars,5,21.881052,five stars wonderful case to carry while trave...
3,BM25,beauty product that is easy to carry while tra...,4,B01MZY5F5B_247706,So convenient and also easy to carry for trave...,5,21.483321,so convenient and also easy to carry for trave...
4,BM25,beauty product that is easy to carry while tra...,5,B01MZY5F5B_247705,So convenient and also easy to carry for trave...,5,21.483321,so convenient and also easy to carry for trave...


In [12]:
# Save labeling view
labeling_view_path = RESULTS_DIR / "final_labeling_view.csv"
labeling_view_df.to_csv(labeling_view_path, index=False)
labeling_view_path

WindowsPath('c:/Users/ruthy/assignments/block6/group_575/DSCI_575_project_omo001_deepray/notebooks/outputs/final_labeling_view.csv')

## 7. Manual relevance labeling

Quantitative retrieval metrics require a small relevance set. Since this project does not use a pre-existing benchmark, relevant `doc_id`s are identified manually by inspecting the top retrieved results for each evaluation query. This keeps the evaluation lightweight and feasible while still allowing a fair comparison across BM25, semantic, and hybrid retrieval.

In [13]:
# Ground-truth relevance judgments for the evaluation queries

relevance_judgments = {
    "face moisturizer": {
        "B01MRR1SAS_591326",
        "B007HLWRTM_154324",
        "B07X9VJCN9_622049",
        "B07MX673RB_241638",
        "B08XMBHL8C_92790",
        "B07FB21B1S_5194",
        "B004JJ6ABG_60982",
        "B072NZ9NN8_68594",
    },
    "something for dry skin": {
        "B01GPKTWJ8_75448",
        "B019BCAMT6_441886",
        "B0148MCDYQ_342012",
        "B01N0EN48U_9167",
        "B00IFZU6W4_281671",
        "B072BP8R6H_475706",
        "B07Q8SWGJ2_583441",
        "B07Q8SWGJ2_583440",
    },
    "product to reduce frizzy hair": {
        "B079DGTY9G_275760",
        "B004BS09WG_47965",
        "B00NGTXOZA_476220",
        "B01195J43I_683572",
        "B07G375Q36_668696",
        "B001W7CRCE_571094",
        "B09V1TFKSK_254875",
    },
    "beauty product that is easy to carry while traveling": {
        "B07BR19ML2_51821",
        "B005FMUBLG_205504",
        "B01MZY5F5B_247706",
        "B01MZY5F5B_247705",
        "B01IAFM4GO_498268",
        "B0BM35X1LF_232794",
    },
    "makeup remover that does not irritate sensitive skin": {
        "B01IAI5GCA_322576",
        "B07TKZRNGZ_175105",
        "B01MTZ0MZN_111210",
        "B09GS65Y1T_264997",
        "B01MXLP1T1_438381",
        "B01FZG0L8Y_680532",
        "B07CT6ZMW6_336066",
    },
}

In [14]:
# Save ground-truth relevance judgments for the evaluation queries
ground_truth_labeling = RESULTS_DIR / "ground_truth_labeling.csv"
labeling_view_df.to_csv(ground_truth_labeling, index=False)
ground_truth_labeling

WindowsPath('c:/Users/ruthy/assignments/block6/group_575/DSCI_575_project_omo001_deepray/notebooks/outputs/ground_truth_labeling.csv')

A small manual relevance set was created before computing precision@5 and recall@5. Documents were labeled as relevant only when they clearly matched the full query intent, not just overlapping keywords. Negative, off-topic, ambiguous, or only partially related results were excluded so that the later metric calculations would reflect genuinely useful retrieval performance.

In [15]:
evaluation_results = evaluate_results(
    results_df=all_results_df,
    relevance_judgments=relevance_judgments,
    k=5,
)

summary_results = summarize_evaluation(evaluation_results)

evaluation_results, summary_results

(   retriever  query_id                                              query  \
 0       BM25         1                                   face moisturizer   
 1       BM25         2                             something for dry skin   
 2       BM25         3                      product to reduce frizzy hair   
 3       BM25         4  beauty product that is easy to carry while tra...   
 4       BM25         5  makeup remover that does not irritate sensitiv...   
 5     Hybrid         1                                   face moisturizer   
 6     Hybrid         2                             something for dry skin   
 7     Hybrid         3                      product to reduce frizzy hair   
 8     Hybrid         4  beauty product that is easy to carry while tra...   
 9     Hybrid         5  makeup remover that does not irritate sensitiv...   
 10  Semantic         1                                   face moisturizer   
 11  Semantic         2                             something fo

In [16]:
# Save evaluation results
evaluation_results_path = RESULTS_DIR / "evaluation_results.csv"
evaluation_results.to_csv(evaluation_results_path, index=False)
evaluation_results_path


WindowsPath('c:/Users/ruthy/assignments/block6/group_575/DSCI_575_project_omo001_deepray/notebooks/outputs/evaluation_results.csv')

## 8 Interpretation of quantitative results

The quantitative evaluation shows that **Hybrid retrieval** achieved the strongest overall performance, with the highest average precision@5 and recall@5 across the evaluation queries. However, the margin over **Semantic retrieval** was small, which suggests that semantic retrieval alone was already quite strong on this query set.

The results also show that **BM25** remained competitive, especially on more direct product-oriented queries, but it performed slightly worse overall than the other two methods. This pattern is consistent with the project design: BM25 benefits from exact lexical overlap, while semantic retrieval captures broader query meaning. Hybrid retrieval combines both signals, which likely explains its small overall advantage.

Overall, these results support the use of **Hybrid retrieval** as the default retrieval method for the project, while also showing that the semantic retriever is a strong standalone baseline.

In [17]:
quantitative_eval_observations = {
    "feature_choice_rationale": (
        "Option 1 was chosen because quantitative evaluation could be added quickly "
        "and meaningfully using custom retrieval metrics without changing the system design."
    ),
    "metric_choice_rationale": (
        "Precision@5 and recall@5 were used because they are explicitly suggested in the "
        "final submission brief and match the project's top-5 retrieval workflow."
    ),
    "labeling_strategy_rationale": (
        "A small manually labeled relevance set was used because it is lightweight, "
        "transparent, and feasible within the scope of the project."
    ),
    "actual_observation": (
        "Hybrid retrieval achieved the best overall average performance, with precision@5 = 0.80 "
        "and recall@5 = 0.5524. Semantic retrieval was nearly identical, while BM25 was slightly weaker overall."
    ),
    "limitations": [
        "The evaluation uses a small manually labeled query set rather than a large benchmark.",
        "Relevance judgments depend on human interpretation.",
        "The metrics evaluate retrieval quality only, not end-to-end answer quality.",
    ],
}